In [1]:
import ee
import geemap
import geopandas as gpd
from shapely.geometry import Point

ee.Initialize(project='omo-turkana-basin')


In [32]:
#load shape file and inspect
gdf=gpd.read_file("C:/Users/Admin/Desktop/KMD/Shapefiles/Kenya_shapes/Kenya/Kenya_region.shp")
gdf.head()

,COUNTIES,COUNT,COUNTRY,AREA_SQKM,geometry
0,BARINGO,2,Kenya,10911.224427,"POLYGON ((35.63043 0.7175, 35.63058 0.71769, 3..."
1,BOMET,2,Kenya,2671.766664,"POLYGON ((35.30328 -0.44208, 35.30389 -0.44202..."
2,BUNGOMA,2,Kenya,3032.528569,"POLYGON ((34.41356 0.81185, 34.41374 0.81232, ..."
3,BUSIA,2,Kenya,1829.481024,"POLYGON ((34.21746 0.28877, 34.21418 0.28824, ..."
4,ELGEYO/MARAKWET,2,Kenya,3017.840876,"POLYGON ((35.68368 1.09633, 35.68366 1.09587, ..."


In [6]:

gdf.crs

<Geographic 2D CRS: EPSG:4210>
Name: Arc 1960
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: Burundi, Kenya, Rwanda, Tanzania and Uganda.
- bounds: (28.85, -11.75, 41.91, 4.63)
Datum: Arc 1960
- Ellipsoid: Clarke 1880 (RGS)
- Prime Meridian: Greenwich

In [7]:
kenya=gdf.dissolve()

In [33]:
#Project to another CRS
kenya = kenya.to_crs(epsg=4326)

kenya.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [9]:
#convert to Google earth Engine
kenya_ee=geemap.geopandas_to_ee(kenya)

In [13]:
#Load Chirps
chirps=ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

In [14]:
#Establish Climatology
baseline = chirps.filterDate(
    '1991-01-01',
    '2020-12-31'
)

In [15]:
# Calculate Annual rainfall totals
years = list(range(1991, 2021))

annual_rainfall = ee.ImageCollection(
    [
        chirps.filterDate(
            f"{year}-01-01",
            f"{year+1}-01-01"
        ).sum()
        .set('year', year)
        for year in years
    ]
)

In [16]:
# Calculate Mean annual rainfall
mean_annual_rainfall = annual_rainfall.mean()

In [23]:
#clip Chirps to shape file
chirps_kenya = mean_annual_rainfall.clip(kenya_ee)

In [29]:
# Load SRTM30 DEM
srtm30 = ee.Image('USGS/SRTMGL1_Ellip/SRTMGL1_Ellip_srtm')

# Clip DEM to Shapefile
srtm30_kenya = srtm30.clip(kenya_ee)

In [30]:
# Load LULC 
lulc = ee.ImageCollection("ESA/WorldCover/v200")

# Filter to latest year and clip to shape file
lulc_kenya = lulc.first().clip(kenya_ee)

In [34]:
# Rainfall statistics
rainfall_stats = mean_annual_rainfall.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=kenya_ee.geometry(),
    scale=5000
)
print(rainfall_stats.getInfo())

{'precipitation_max': 2351.6658610562486, 'precipitation_min': 20.427950856013922}


In [35]:
#Classify rainfall
rainfall_classified = mean_annual_rainfall.where(
    mean_annual_rainfall.lt(500), 1
).where(
    mean_annual_rainfall.lt(1000), 2
).where(
    mean_annual_rainfall.lt(1500), 3
).where(
    mean_annual_rainfall.lt(2000), 4
).where(
    mean_annual_rainfall.gte(2000), 5
)


In [36]:
# Calculate slope from elevation
slope = ee.Terrain.slope(srtm30_kenya)

# slope statistics
slope_stats = slope.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=kenya_ee.geometry(),
    scale=30
)
print(slope_stats.getInfo())

EEException: Image.load: Image asset 'USGS/SRTMGL1_Ellip/SRTMGL1_Ellip_srtm' not found (does not exist or caller does not have access).